In [5]:
%pip install langchain langchain_groq serpapi

Looking in indexes: https://sres.web.boeing.com/artifactory/api/pypi/pypi-releases/simple
Note: you may need to restart the kernel to use updated packages.


### Chain-of-Thought (CoT)
Concept:

CoT encourages the model to “think step by step.”

Example:

Task: Solve 35 + 47 – 12

Prompt: Let's solve step by step:
1. Add 35 and 47.
2. Subtract 12 from the result.
Answer:

Expected Output:
    Step 1: 35 + 47 = 82
Step 2: 82 – 12 = 70
Answer: 70

In [ ]:
import os

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
load_dotenv()

# Initialize using ChatGroq
llm = ChatGroq(
    #model="llama-3.1-8b-instant", 
    model="openai/gpt-oss-20b",
    temperature=0.7,
    api_key=os.getenv("GROQ_API_KEY")
)
prompt = PromptTemplate(template="Let's solve step by step: {problem}\nAnswer:", input_variables=["problem"])
chain = prompt | llm


print(chain.invoke({"problem": "35 + 47 - 12"}))

content='**Step 1:** Add the first two numbers  \n\\(35 + 47 = 82\\)\n\n**Step 2:** Subtract the last number  \n\\(82 - 12 = 70\\)\n\n**Answer:** 70' additional_kwargs={'reasoning_content': 'We need to solve step by step: 35 + 47 - 12. We can do 35+47=82, then 82-12=70. So answer is 70. Probably want step-by-step explanation. So answer: 70.'} response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 88, 'total_tokens': 199, 'completion_time': 0.116008116, 'completion_tokens_details': {'reasoning_tokens': 55}, 'prompt_time': 0.004269469, 'prompt_tokens_details': None, 'queue_time': 0.05295804, 'total_time': 0.120277585}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e0f4f3edeb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d81ab-cbda-7f40-9624-fa0c1f40b512-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 88, 'output_tokens': 111, 'total_tokens': 199, 'output_toke

### ReAct (Reason + Act)
Concept:

Combine reasoning traces with actions. The model alternates between Thought, Action, Observation.

Example:

Task: Find the current CEO of OpenAI and summarize their background.

Prompt:
Question: Who is the current CEO of OpenAI?


Thought: I should look up the current CEO.
Action: Search("CEO of OpenAI")
Observation: (search result)


Thought: Now summarize their background.
Action: Summarize

In [ ]:
from langchain_groq import ChatGroq

from langchain_community.agent_toolkits.load_tools import load_tools

from langgraph.prebuilt import create_react_agent

import os

# Set API keys

os.environ["SERPAPI_API_KEY"] = "5c7b96ec00f71e29e7ec068d8c54a4f717f80220dbc1149fc5c63a76bef6f98a"

os.environ["GROQ_API_KEY"] = "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"

# 1. Setup LLM

llm = ChatGroq(model="openai/gpt-oss-120b", api_key=os.environ["GROQ_API_KEY"])

# 2. Load tools

tools = load_tools(["serpapi"])

# 3. Create agent

agent_executor = create_react_agent(llm, tools)

# 4. ✅ Take input from user

#query = input("Ask your question: ")
query = input("Ask your question: ")
#query = "Who is the CEO of OpenAI and summarize their background?"
 
# 5. Run

for chunk in agent_executor.stream({"messages": [("human", query)]}):

    print(chunk)
 

Tree-of-Thought (ToT)
Concept:

Instead of a single chain, explore multiple possible reasoning paths (branches) and evaluate the best one.

Example:

Task: Brainstorm 3 startup ideas for AI in healthcare and pick the best one.

Prompt:

Let's think in multiple paths:
Path 1: ...
Path 2: ...
Path 3: ...
Evaluate and choose the best idea with reasoning.

In [ ]:
ideas = llm.invoke("Generate 3 startup ideas for AI in healthcare")
evaluations = llm.invoke(f"Evaluate the following ideas and pick the most impactful one: {ideas}")
print(evaluations)

### Graph-of-Thought (GoT)
Concept:

Generalizes ToT into a graph: multiple reasoning paths can converge, reuse sub-results, or feed into each other.

Example:

Task: Plan a week-long AI learning curriculum.

Node 1: Day 1 topics

Node 2: Day 2 topics (depends on Node 1)

Node 3: Assessment (depends on Node 2)

This allows iterative refinement: if Day 2 is too heavy, adjust Day 1.

In [ ]:
# Basic example of building a graph of reasoning nodes
thoughts = {
"Day 1": "Introduction to AI + history",
"Day 2": "Machine Learning basics",
"Day 3": "Deep Learning intro",
"Assessment": "Quiz based on Day 1-3"
}


for node, content in thoughts.items():print(f"{node} -> {content}")

    Summary: When to Use What
    Technique	When to Use
    CoT	Straightforward reasoning tasks, math problems, logical puzzles.
    ReAct	When interaction with tools, APIs, or dynamic data is needed.
    ToT	Open-ended reasoning, brainstorming, planning where multiple paths need exploration.
    GoT	Very complex workflows, research planning, knowledge graph reasoning, collaborative multi-step problem solving.